# Predictive Modeling (Y1 -> Y2)

**Y1 → Y2 out-of-time generalization.** Ridge is fit on Y1 (driving from Interval 0, health at Interval 1) with the regularization strength tuned by inner 10-fold CV on the Y1 training data only, then evaluated on Y2 (driving from Interval 1, health at Interval 2). For every outcome, four models are compared on per-subject squared error in Y2:

* `full`     – driving variables + demographics
* `driving`  – driving variables only
* `demo`  – demographics only
* `baseline` – constant prediction = mean of the Y1 training outcome

Four paired one-tailed comparisons (in the direction of better prediction) probe the substantive questions:

| Comparison | Question it answers |
|---|---|
| `driving`  vs `baseline` | Does driving predict at all? |
| `demo`  vs `baseline` | Do demographics predict at all? |
| `full`     vs `demo`  | Does driving add anything beyond demographics? |
| `full`     vs `driving`  | Do demographics add anything beyond driving? |

For each comparison we report a paired t-test on per-subject squared-error differences and a sign-flip permutation test (10,000 permutations). p-values are FDR-corrected within each comparison family across the 12 outcomes.

**Outcome scale.** Outcomes are kept on their native PROMIS / life-satisfaction scale; MSEs and effect sizes are interpretable in the original units.

**Missing-data policy.**
* Subjects with missing `y` in either Y1 (train) or Y2 (test) are dropped — squared error is undefined without a true value.
* For predictor columns, the sklearn Pipeline imputes missing values (mean for numeric, mode for categorical). The imputer is fit on Y1 only; when `Pipeline.predict()` is called on Y2 it uses `.transform()`, which applies the Y1-fit statistics. Y2 subjects with missing predictors therefore receive Y1 column means/modes for those features. 

In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from statsmodels.stats.multitest import multipletests

In [2]:
ivs = ['DaysDriving','Miles_n','Trips','TripMinutes_n','TripChains',
       'MilesPerTrip_n','MinutesPerTrip_n','MilesPerChain_n','MinutesPerChain_n',
       'Average_speed','LeftTurnCount','RightToLeftTurnRatio_n',
       'TripsAMPeak','PercentTripsAMPeak_n','TripsPMPeak','PercentTripsPMPeak_n',
       'TripsAtNight','PercentTripsAtNight_n','TripsLt15Miles','PercentDistLt15Miles_n',
       'TripsVgt60','PercentTripsVgt60_n','SpeedingPer1000Miles','DecelerationPer1000Miles']

dvs = ['LIFESATISFACTION','cog','phys','fati','socialrole','iso','info','emo','ins','dep','anx','ang']
demos_cat = ['Site','GENDER','RACE_ETH','WORK','MARRIAGE']
demos_num = ['Age','EDUCATION','INCOME']  # ordinal-coded but treated as numeric for consistency with R analyses

In [3]:
y1 = pd.read_csv('../data/y1_subject_level.csv')
y2 = pd.read_csv('../data/y2_subject_level.csv')

for df in (y1, y2):
    for col in ['XID'] + demos_cat:
        if col in df.columns:
            df[col] = df[col].astype('category')
# Outcomes are NOT scaled: MSE is reported on the original PROMIS / life-satisfaction scale.

print('Y1 file rows =', len(y1))
print('Y2 file rows =', len(y2))
print('Subjects in both Y1 and Y2:',
      len(set(y1['XID'].astype(str)).intersection(set(y2['XID'].astype(str)))))

Y1 file rows = 2990
Y2 file rows = 2990
Subjects in both Y1 and Y2: 2990


In [4]:
def build_preprocessor(numeric_cols, categorical_cols):
    """Per-column-type preprocessor with imputation.
    Numeric: mean-impute -> standard-scale.
    Categorical: mode-impute -> one-hot (drop first; handle_unknown='ignore').
    All imputers and the scaler are fit on training data only; .transform() on
    Y2 therefore uses Y1-derived statistics to fill in any missing Y2 predictors."""
    numeric_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='mean')),
        ('scale',  StandardScaler())])
    categorical_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe',    OneHotEncoder(drop='first', handle_unknown='ignore'))])
    transformers = []
    if numeric_cols:
        transformers.append(('num', numeric_pipe, numeric_cols))
    if categorical_cols:
        transformers.append(('cat', categorical_pipe, categorical_cols))
    return ColumnTransformer(transformers)

In [5]:
def train_y1_predict_y2(y1_df, y2_df, dv, id_col,
                        numeric_ivs, categorical_ivs,
                        alphas=None, random_state=0, inner_splits=10):
    """Fit Ridge on Y1 with inner 10-fold CV for alpha; predict on Y2.

    Preprocessing notes:
      * The imputers and the scaler inside the Pipeline are fit on Y1 only.
      * At predict time, Pipeline.predict() calls .transform() on Y2, which
        applies the Y1-fit statistics. Any Y2 subject missing a numeric
        predictor receives the Y1 column mean for that predictor; any Y2
        subject missing a categorical predictor receives the Y1 mode.
      * Subjects with missing y are dropped from train/test as appropriate
        (no squared error is defined without a true value).

    Returns one row per Y2 subject with non-missing y: id, y_true, y_pred,
    squared error, and the chosen alpha (constant within a single Y1 fit)."""
    if alphas is None:
        alphas = np.logspace(-2, 5, 100)

    y1_use = y1_df.dropna(subset=[dv]).copy()
    y2_use = y2_df.dropna(subset=[dv]).copy()

    cols = list(numeric_ivs) + list(categorical_ivs)
    X_train = y1_use[cols]; y_train = y1_use[dv].values
    X_test  = y2_use[cols]; y_test  = y2_use[dv].values
    ids_test = y2_use[id_col].values

    preprocessor = build_preprocessor(list(numeric_ivs), list(categorical_ivs))
    inner_cv = KFold(n_splits=inner_splits, shuffle=True, random_state=random_state)
    model = Pipeline([
        ('pre', preprocessor),
        ('reg', RidgeCV(alphas=alphas, scoring='neg_mean_squared_error', cv=inner_cv))])
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    alpha_chosen = model.named_steps['reg'].alpha_

    return pd.DataFrame({
        'subject': ids_test,
        'y_true':  y_test,
        'y_pred':  y_pred,
        'sq_err':  (y_test - y_pred) ** 2,
        'alpha':   alpha_chosen,
    })


def baseline_y1_predict_y2(y1_df, y2_df, dv, id_col):
    """Constant-prediction baseline: predict every Y2 subject as the Y1
    training mean of the outcome. This is the 'guessing' baseline R1 asked for."""
    y1_use = y1_df.dropna(subset=[dv])
    y2_use = y2_df.dropna(subset=[dv]).copy()
    mu = y1_use[dv].mean()
    y_test = y2_use[dv].values
    return pd.DataFrame({
        'subject': y2_use[id_col].values,
        'y_true':  y_test,
        'y_pred':  mu,
        'sq_err':  (y_test - mu) ** 2,
    })


def paired_perm_test_mean_diff(diff, n_perm=10000, alternative='less',
                               random_state=0):
    """Permutation test on the mean of paired differences via sign-flipping.
    Under H0 of zero-symmetric differences, signs are exchangeable."""
    rng = np.random.default_rng(random_state)
    d = np.asarray(diff)
    d = d[~np.isnan(d)]
    obs = d.mean()
    n = len(d)
    signs = rng.choice([-1, 1], size=(n_perm, n))
    null = (signs * d).mean(axis=1)
    if alternative == 'less':
        p = (np.sum(null <= obs) + 1) / (n_perm + 1)
    elif alternative == 'greater':
        p = (np.sum(null >= obs) + 1) / (n_perm + 1)
    else:
        p = (np.sum(np.abs(null) >= abs(obs)) + 1) / (n_perm + 1)
    return obs, p


def imputation_footprint(y2_df, dv, predictor_cols):
    """For each outcome, count Y2 subjects (those with non-missing y) who have
    at least one missing predictor. Reports the per-column NaN count too so
    the manuscript can summarize the imputation footprint transparently."""
    y2_use = y2_df.dropna(subset=[dv])
    sub = y2_use[list(predictor_cols)]
    n_total = len(sub)
    any_missing = int(sub.isna().any(axis=1).sum())
    return {
        'dv': dv,
        'n_y2_with_y': int(n_total),
        'n_y2_any_imputed_predictor': any_missing,
        'pct_any_imputed_predictor': float(any_missing / n_total * 100) if n_total else 0.0,
        'per_column_missing': sub.isna().sum().to_dict(),
    }

In [6]:
def run_y1y2_compare(y1_df, y2_df, dv, id_col,
                     numeric_full, categorical_full,
                     numeric_demo, categorical_demo,
                     numeric_driving, categorical_driving,
                     alphas=None, save_path=None):
    """Fit four models on Y1, evaluate on Y2, and run paired tests on per-subject
    squared error.

    Models:
      full     = driving + demographics
      driving  = driving only
      demo  = demographics only
      baseline = constant = Y1 training mean

    Substantive paired comparisons (one-tailed, smaller error in first model):
      driving vs baseline -- does driving predict at all?
      demo vs baseline -- do demographics predict at all?
      full    vs demo  -- does driving add beyond demographics?
      full    vs driving  -- do demographics add beyond driving?
    """
    full_res     = train_y1_predict_y2(y1_df, y2_df, dv, id_col,
                                       numeric_full,    categorical_full,    alphas=alphas)
    driving_res  = train_y1_predict_y2(y1_df, y2_df, dv, id_col,
                                       numeric_driving, categorical_driving, alphas=alphas)
    demo_res  = train_y1_predict_y2(y1_df, y2_df, dv, id_col,
                                       numeric_demo, categorical_demo, alphas=alphas)
    baseline_res = baseline_y1_predict_y2(y1_df, y2_df, dv, id_col)

    merged = (full_res.rename(columns={'sq_err': 'sq_err_full'})[['subject', 'sq_err_full']]
              .merge(driving_res.rename(columns={'sq_err': 'sq_err_driving'})[['subject', 'sq_err_driving']],   on='subject')
              .merge(demo_res.rename(columns={'sq_err': 'sq_err_demo'})[['subject', 'sq_err_demo']],   on='subject')
              .merge(baseline_res.rename(columns={'sq_err': 'sq_err_baseline'})[['subject', 'sq_err_baseline']],on='subject'))

    comparisons = [
        ('driving',  'baseline'),
        ('demo',  'baseline'),
        ('full',     'demo'),
        ('full',     'driving'),
    ]

    summary = {
        'dv': dv,
        'n_y2': int(merged.shape[0]),
        'mse_full':     float(merged['sq_err_full'].mean()),
        'mse_driving':  float(merged['sq_err_driving'].mean()),
        'mse_demo':  float(merged['sq_err_demo'].mean()),
        'mse_baseline': float(merged['sq_err_baseline'].mean()),
        'alpha_full':    float(full_res['alpha'].iloc[0]),
        'alpha_driving': float(driving_res['alpha'].iloc[0]),
        'alpha_demo': float(demo_res['alpha'].iloc[0]),
    }

    print(f"\n=== Y1 -> Y2 generalization: DV = {dv} (N_Y2 = {summary['n_y2']}) ===")
    print(f"  MSE_full     = {summary['mse_full']:.4f}")
    print(f"  MSE_driving  = {summary['mse_driving']:.4f}")
    print(f"  MSE_demo  = {summary['mse_demo']:.4f}")
    print(f"  MSE_baseline = {summary['mse_baseline']:.4f}")

    for a, b in comparisons:
        diff = merged[f'sq_err_{a}'] - merged[f'sq_err_{b}']
        diff_valid = diff.dropna()
        t_stat, p_t = stats.ttest_1samp(diff_valid, 0, alternative='less')
        obs, p_perm = paired_perm_test_mean_diff(diff_valid.values, alternative='less')
        summary[f't_{a}_vs_{b}']      = float(t_stat)
        summary[f'p_{a}_vs_{b}']      = float(p_t)
        summary[f'perm_p_{a}_vs_{b}'] = float(p_perm)
        summary[f'mean_diff_{a}_vs_{b}'] = float(obs)
        print(f"  {a:<7s} vs {b:<8s}: mean diff = {obs:+.4f}, t = {t_stat:.3f}, t-p = {p_t:.4g}, perm-p = {p_perm:.4g}")

    if save_path:
        merged.to_csv(f'{save_path}/{dv}_per_subject.csv', index=False)

    return merged, summary

In [7]:
# Single wide log-spaced grid per DV. the wide range lets RidgeCV find a good value. Watch for chosen alphas pinned at
# the grid boundaries in the summary -- that would indicate the grid is too narrow.
alpha_grids = {dv: np.logspace(-2, 5, 100) for dv in dvs}

# Predictor sets for the four models.
full_num, full_cat       = ivs + demos_num, demos_cat
driving_num, driving_cat = ivs,             []
demos_num, demos_cat = demos_num,       demos_cat
# baseline uses no predictors

y1y2_per_subject = {}
y1y2_summary = []
imputation_rows = []

# save per-subject squared errors
PER_SUBJECT_DIR = 'output/ridge_errors/'
os.makedirs(PER_SUBJECT_DIR, exist_ok=True)

for dv in dvs:
    per_sub, summary = run_y1y2_compare(
        y1, y2, dv, id_col='XID',
        numeric_full=full_num,       categorical_full=full_cat,
        numeric_demo=demos_num, categorical_demo=demos_cat,
        numeric_driving=driving_num, categorical_driving=driving_cat,
        alphas=alpha_grids[dv],
        save_path=PER_SUBJECT_DIR)
    y1y2_per_subject[dv] = per_sub
    y1y2_summary.append(summary)

    # Imputation footprint for the full-model predictor set (the broadest)
    imp = imputation_footprint(y2, dv, full_num + full_cat)
    imputation_rows.append(imp)

summary_df = pd.DataFrame(y1y2_summary)

# FDR-correct within each comparison family across the 12 DVs
comparison_pairs = [('driving','baseline'),
                    ('demo','baseline'),
                    ('full','demo'),
                    ('full','driving')]
for a, b in comparison_pairs:
    for prefix in ('p', 'perm_p'):
        col = f'{prefix}_{a}_vs_{b}'
        summary_df[col + '_fdr'] = multipletests(summary_df[col].values, method='fdr_by')[1]

summary_df.to_csv(f'output/ridge_summary.csv', index=False)
summary_df


=== Y1 -> Y2 generalization: DV = LIFESATISFACTION (N_Y2 = 2704) ===
  MSE_full     = 0.3448
  MSE_driving  = 0.3802
  MSE_demo  = 0.3548
  MSE_baseline = 0.4000
  driving vs baseline: mean diff = -0.0198, t = -5.542, t-p = 1.637e-08, perm-p = 9.999e-05
  demo    vs baseline: mean diff = -0.0452, t = -9.672, t-p = 4.466e-22, perm-p = 9.999e-05
  full    vs demo    : mean diff = -0.0100, t = -3.782, t-p = 7.935e-05, perm-p = 0.0002
  full    vs driving : mean diff = -0.0354, t = -8.613, t-p = 5.962e-18, perm-p = 9.999e-05

=== Y1 -> Y2 generalization: DV = cog (N_Y2 = 2702) ===
  MSE_full     = 0.3915
  MSE_driving  = 0.4097
  MSE_demo  = 0.3894
  MSE_baseline = 0.4282
  driving vs baseline: mean diff = -0.0185, t = -4.916, t-p = 4.677e-07, perm-p = 9.999e-05
  demo    vs baseline: mean diff = -0.0387, t = -9.256, t-p = 2.101e-20, perm-p = 9.999e-05
  full    vs demo    : mean diff = +0.0021, t = 1.094, t-p = 0.8629, perm-p = 0.8602
  full    vs driving : mean diff = -0.0181, t = -5.47

,dv,n_y2,mse_full,mse_driving,mse_demo,mse_baseline,alpha_full,alpha_driving,alpha_demo,t_driving_vs_baseline,...,perm_p_full_vs_driving,mean_diff_full_vs_driving,p_driving_vs_baseline_fdr,perm_p_driving_vs_baseline_fdr,p_demo_vs_baseline_fdr,perm_p_demo_vs_baseline_fdr,p_full_vs_demo_fdr,perm_p_full_vs_demo_fdr,p_full_vs_driving_fdr,perm_p_full_vs_driving_fdr
0,LIFESATISFACTION,2704,0.344791,0.380206,0.354792,0.399960,107.226722,91.116276,126.185688,-5.542264,...,0.0001,-0.035414,6.096335e-07,0.001241,1.663015e-20,0.000372,0.001477,0.003723,2.220254e-16,0.000414
1,cog,2702,0.391529,0.409656,0.389444,0.428151,5.722368,10.974988,5.722368,-4.916174,...,0.0001,-0.018127,5.805117e-06,0.001241,3.912044e-19,0.000372,1.000000,1.000000,1.636626e-07,0.000414
2,phys,2690,0.393042,0.413721,0.412947,0.434441,47.508102,174.752840,242.012826,-4.956819,...,0.0001,-0.020680,5.805117e-06,0.001241,1.740896e-08,0.000372,0.000026,0.003723,1.691454e-08,0.000414
3,fati,2700,0.571743,0.584903,0.579936,0.592251,107.226722,394.420606,394.420606,-2.983557,...,0.0001,-0.013160,1.070505e-02,0.013405,1.002987e-06,0.000372,0.009810,0.011170,1.636626e-07,0.000414
4,socialrole,2659,0.426705,0.431007,0.430931,0.436542,107.226722,91.116276,464.158883,-1.928121,...,0.0015,-0.004302,1.434919e-01,0.156386,1.934488e-04,0.000372,0.580657,0.623683,3.544173e-03,0.004654
5,iso,2687,0.400445,0.406457,0.401369,0.409938,394.420606,890.215085,107.226722,-1.777705,...,0.0013,-0.006012,1.758717e-01,0.186174,1.159826e-03,0.001862,1.000000,1.000000,3.544173e-03,0.004400
6,info,2695,0.628903,0.664207,0.628241,0.669345,29.150531,91.116276,12.915497,-1.392473,...,0.0001,-0.035305,2.854158e-01,0.278584,7.629730e-10,0.000372,1.000000,1.000000,6.839831e-10,0.000414
7,emo,2698,0.519382,0.536131,0.519907,0.540764,77.426368,242.012826,29.150531,-1.506718,...,0.0001,-0.016749,2.730828e-01,0.268504,4.932872e-06,0.000372,1.000000,1.000000,5.829087e-06,0.000414
8,ins,2647,0.578791,0.645928,0.577953,0.660061,10.974988,174.752840,6.734151,-3.348620,...,0.0001,-0.067137,3.833352e-03,0.003723,5.239121e-14,0.000372,1.000000,1.000000,1.832163e-11,0.000414
9,dep,2702,0.184730,0.186830,0.185546,0.188403,1232.846739,1232.846739,890.215085,-2.267132,...,0.0001,-0.002101,7.280413e-02,0.077573,1.333551e-05,0.000372,0.643785,0.624800,7.141295e-05,0.000414
